# Bước tiếp theo: Chọn đặc trưng để dự báo mực nước lũ

Notebook này được viết để **nối tiếp** `processing_data_lake.ipynb`.

Mục tiêu:
1. Dùng dữ liệu hồ đã tiền xử lý (`lake_daily`) để tạo bài toán dự báo `water_level`.
2. Sinh thêm đặc trưng chuỗi thời gian (lag, rolling, seasonal, hydrology features).
3. Chấm điểm đặc trưng bằng nhiều cách: correlation, mutual information, tree importance.
4. Chọn ra bộ đặc trưng cuối cùng để huấn luyện mô hình dự báo.

> Gợi ý thực hành: dùng `horizon = 1` để dự báo mực nước ngày kế tiếp. Sau khi ổn, có thể thử `horizon = 3` hoặc `horizon = 7`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

## 1) Nạp `lake_daily`

- Nếu notebook cũ đã chạy tới phần EDA thì biến `lake_daily` đã tồn tại.
- Nếu chưa có, cell dưới sẽ dựng lại `lake_daily` từ file CSV theo đúng logic gần giống notebook cũ.

In [ ]:
if 'lake_daily' not in globals():
    df = pd.read_csv('/mnt/data/water_data_full_combined.csv', low_memory=False)

    lake = df[df['type'].str.lower() == 'lake'].copy()
    lake['timestamp'] = pd.to_datetime(lake['Thời gian (UTC)'], errors='coerce')
    lake = lake.sort_values(['Mã trạm/LakeCode', 'timestamp']).reset_index(drop=True)

    lake = lake.rename(columns={
        'Mã trạm/LakeCode': 'lake_code',
        'Trạm/Hồ': 'lake_name',
        'Tên sông/Lưu vực': 'basin_name',
        'Tên tỉnh': 'province',
        'Mực nước (m)': 'water_level',
        'Dung tích (m3)': 'storage',
        'Dung tích TK (m3)': 'designed_storage',
        'Tỷ lệ dung tích (%)': 'storage_pct',
        'Q đến (m3/s)': 'inflow',
        'Q xả (m3/s)': 'outflow',
        'Mực nước BT (m)': 'normal_level',
        'Mực nước GC (m)': 'dead_level',
        'Cảnh báo/Xu thế': 'trend_text'
    })

    num_cols = [
        'water_level', 'storage', 'designed_storage', 'storage_pct',
        'inflow', 'outflow', 'normal_level', 'dead_level',
        'x', 'y', 'province_code', 'basin_code'
    ]

    for c in num_cols:
        if c in lake.columns:
            lake[c] = pd.to_numeric(lake[c], errors='coerce')

    lake_daily = (
        lake.set_index('timestamp')
            .groupby('lake_code')
            .resample('D')
            .agg({
                'lake_name': 'last',
                'province': 'last',
                'basin_name': 'last',
                'water_level': 'mean',
                'inflow': 'mean',
                'outflow': 'mean',
                'storage': 'last',
                'designed_storage': 'last',
                'storage_pct': 'last',
                'normal_level': 'last',
                'dead_level': 'last',
                'trend_text': 'last',
                'x': 'last',
                'y': 'last',
                'province_code': 'last',
                'basin_code': 'last'
            })
            .reset_index()
    )

    def fill_time_series(g):
        g = g.sort_values('timestamp').copy()

        for col in ['water_level', 'storage', 'storage_pct']:
            if col in g.columns:
                g[col] = g[col].interpolate(method='linear', limit=3, limit_direction='both')

        for col in ['inflow', 'outflow']:
            if col in g.columns:
                g[f'{col}_missing'] = g[col].isna().astype(int)
                g[col] = g[col].interpolate(method='linear', limit=2, limit_direction='both')

        for col in ['normal_level', 'dead_level', 'x', 'y', 'province_code', 'basin_code']:
            if col in g.columns:
                g[col] = g[col].ffill().bfill()

        return g

    lake_daily = (
        lake_daily.groupby('lake_code', group_keys=False)
                  .apply(fill_time_series)
                  .reset_index(drop=True)
    )

lake_daily = lake_daily.sort_values(['lake_code', 'timestamp']).copy()
print(lake_daily.shape)
lake_daily.head()

## 2) Tạo đặc trưng mới cho bài toán dự báo

Tư tưởng chính:
- **Lag features**: hồ ngày mai thường phụ thuộc mạnh vào hồ hôm nay và vài ngày trước.
- **Rolling features**: làm mượt nhiễu, giữ xu hướng ngắn hạn / trung hạn.
- **Hydrology features**: `net_inflow`, `storage_ratio`, khoảng cách tới ngưỡng bình thường/chết.
- **Seasonal features**: tháng trong năm, chu kỳ mùa.

Lưu ý:
- Các rolling statistics được tính trên chuỗi đã `shift(1)` để giảm rủi ro leakage.

In [ ]:
feature_data = lake_daily.copy()

feature_data['month'] = feature_data['timestamp'].dt.month
feature_data['dayofyear'] = feature_data['timestamp'].dt.dayofyear

feature_data['month_sin'] = np.sin(2 * np.pi * feature_data['month'] / 12)
feature_data['month_cos'] = np.cos(2 * np.pi * feature_data['month'] / 12)
feature_data['doy_sin'] = np.sin(2 * np.pi * feature_data['dayofyear'] / 366)
feature_data['doy_cos'] = np.cos(2 * np.pi * feature_data['dayofyear'] / 366)

feature_data['net_inflow'] = feature_data['inflow'] - feature_data['outflow']
feature_data['storage_gap'] = feature_data['designed_storage'] - feature_data['storage']
feature_data['storage_ratio'] = feature_data['storage'] / feature_data['designed_storage']
feature_data['level_above_normal'] = feature_data['water_level'] - feature_data['normal_level']
feature_data['level_above_dead'] = feature_data['water_level'] - feature_data['dead_level']

# lag features
for lag in [1, 2, 3, 7, 14, 30]:
    for col in ['water_level', 'inflow', 'outflow', 'storage_pct', 'net_inflow']:
        feature_data[f'{col}_lag{lag}'] = feature_data.groupby('lake_code')[col].shift(lag)

# rolling features (dùng dữ liệu quá khứ)
for win in [3, 7, 14]:
    shifted_level = feature_data.groupby('lake_code')['water_level'].shift(1)
    feature_data[f'water_level_roll_mean_{win}'] = (
        shifted_level.groupby(feature_data['lake_code']).transform(lambda s: s.rolling(win).mean())
    )
    feature_data[f'water_level_roll_std_{win}'] = (
        shifted_level.groupby(feature_data['lake_code']).transform(lambda s: s.rolling(win).std())
    )

for win in [3, 7]:
    shifted_inflow = feature_data.groupby('lake_code')['inflow'].shift(1)
    shifted_outflow = feature_data.groupby('lake_code')['outflow'].shift(1)

    feature_data[f'inflow_roll_mean_{win}'] = (
        shifted_inflow.groupby(feature_data['lake_code']).transform(lambda s: s.rolling(win).mean())
    )
    feature_data[f'outflow_roll_mean_{win}'] = (
        shifted_outflow.groupby(feature_data['lake_code']).transform(lambda s: s.rolling(win).mean())
    )

print(feature_data.shape)
feature_data.head()

## 3) Tạo target cần dự báo

Ví dụ dưới đây dùng:
- `horizon = 1`: dự báo mực nước ngày kế tiếp.

Có thể đổi thành:
- `horizon = 3`: dự báo sau 3 ngày.
- `horizon = 7`: dự báo sau 7 ngày.

In [ ]:
horizon = 1
feature_data[f'target_water_level_t_plus_{horizon}'] = (
    feature_data.groupby('lake_code')['water_level'].shift(-horizon)
)

feature_data[[
    'lake_code', 'timestamp', 'water_level', f'target_water_level_t_plus_{horizon}'
]].head(10)

## 4) Khai báo danh sách biến ứng viên

Ta chia làm 4 nhóm:
1. **Biến đo trực tiếp**: water level, inflow, outflow, storage...
2. **Biến động lực học**: lag, rolling, delta.
3. **Biến ngữ cảnh hồ**: normal level, dead level, designed storage.
4. **Biến mùa vụ**: month/doy sine-cosine.

In [ ]:
# Các biến gốc / cơ sở
base_features = [
    'water_level', 'inflow', 'outflow', 'storage', 'designed_storage', 'storage_pct',
    'normal_level', 'dead_level', 'inflow_missing', 'outflow_missing',
    'net_inflow', 'storage_gap', 'storage_ratio',
    'level_above_normal', 'level_above_dead',
    'month_sin', 'month_cos', 'doy_sin', 'doy_cos'
]

# Nếu notebook trước đã có delta / rolling thì lấy lại, nếu chưa có thì bỏ qua
optional_features = [
    'delta_level_1d', 'delta_level_3d', 'rolling_mean_3', 'rolling_std_7'
]
optional_features = [c for c in optional_features if c in feature_data.columns]

lag_roll_features = [
    c for c in feature_data.columns
    if any(c.startswith(prefix) for prefix in [
        'water_level_lag', 'inflow_lag', 'outflow_lag',
        'storage_pct_lag', 'net_inflow_lag',
        'water_level_roll_mean_', 'water_level_roll_std_',
        'inflow_roll_mean_', 'outflow_roll_mean_'
    ])
]

candidate_features = [c for c in base_features if c in feature_data.columns] + optional_features + lag_roll_features
candidate_features = list(dict.fromkeys(candidate_features))  # bỏ trùng

len(candidate_features), candidate_features[:20]

## 5) Tạo tập dữ liệu model-ready

Ở bước này:
- bỏ các dòng chưa có target,
- giữ lại các cột cần thiết,
- nội suy thiếu bằng median theo từng hồ,
- sẵn sàng cho feature selection.

In [ ]:
target_col = f'target_water_level_t_plus_{horizon}'

model_df = feature_data[['lake_code', 'lake_name', 'timestamp', target_col] + candidate_features].copy()
model_df = model_df.dropna(subset=[target_col]).copy()

# Loại các dòng thiếu quá nhiều biến
max_missing_allowed = int(len(candidate_features) * 0.35)
model_df = model_df[model_df[candidate_features].isna().sum(axis=1) <= max_missing_allowed].copy()

# Fill NA theo median trong từng hồ, sau đó fallback sang median toàn cục
for col in candidate_features:
    model_df[col] = model_df.groupby('lake_code')[col].transform(lambda s: s.fillna(s.median()))
    model_df[col] = model_df[col].fillna(model_df[col].median())

print(model_df.shape)
print(model_df[candidate_features].isna().sum().sum())
model_df.head()

## 6) Chia train/validation theo thời gian

Không chia random toàn bộ chuỗi thời gian, vì sẽ làm rò rỉ thông tin tương lai.
Ở đây dùng ngưỡng thời gian 80/20:
- 80% đầu: train
- 20% cuối: validation

In [ ]:
cutoff_time = model_df['timestamp'].quantile(0.8)
train_df = model_df[model_df['timestamp'] <= cutoff_time].copy()
valid_df = model_df[model_df['timestamp'] > cutoff_time].copy()

X_train = train_df[candidate_features]
y_train = train_df[target_col]
X_valid = valid_df[candidate_features]
y_valid = valid_df[target_col]

print('cutoff_time =', cutoff_time)
print('train shape =', X_train.shape)
print('valid shape =', X_valid.shape)

## 7) Chấm điểm đặc trưng theo nhiều phương pháp

Ta dùng 3 nhóm kỹ thuật:
1. **Pearson correlation**: đo quan hệ tuyến tính.
2. **Mutual Information**: bắt được quan hệ phi tuyến.
3. **Tree importance** (ExtraTrees): đánh giá đóng góp trong mô hình phi tuyến.

Sau đó lấy **trung bình thứ hạng** để ra danh sách ổn định hơn.

In [ ]:
# Để chạy nhanh hơn, có thể lấy mẫu train
train_sample = train_df.sample(n=min(20000, len(train_df)), random_state=42)
valid_sample = valid_df.sample(n=min(8000, len(valid_df)), random_state=42)

X_train_sample = train_sample[candidate_features]
y_train_sample = train_sample[target_col]
X_valid_sample = valid_sample[candidate_features]
y_valid_sample = valid_sample[target_col]

pearson_abs = pd.Series({
    col: train_df[col].corr(train_df[target_col])
    for col in candidate_features
}, dtype='float64').abs().sort_values(ascending=False)

mi_scores = pd.Series(
    mutual_info_regression(X_train_sample, y_train_sample, random_state=42),
    index=candidate_features
).sort_values(ascending=False)

tree_model = ExtraTreesRegressor(
    n_estimators=160,
    max_depth=12,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42
)
tree_model.fit(X_train_sample, y_train_sample)

y_pred = tree_model.predict(X_valid_sample)
print('Validation MAE =', round(mean_absolute_error(y_valid_sample, y_pred), 4))
print('Validation R2  =', round(r2_score(y_valid_sample, y_pred), 4))

tree_importance = pd.Series(
    tree_model.feature_importances_,
    index=candidate_features
).sort_values(ascending=False)

feature_scores = pd.DataFrame({
    'pearson_abs': pearson_abs,
    'mutual_info': mi_scores,
    'tree_importance': tree_importance
}).fillna(0)

for col in ['pearson_abs', 'mutual_info', 'tree_importance']:
    feature_scores[f'rank_{col}'] = feature_scores[col].rank(ascending=False, method='average')

feature_scores['avg_rank'] = feature_scores[[
    'rank_pearson_abs', 'rank_mutual_info', 'rank_tree_importance'
]].mean(axis=1)

feature_scores = feature_scores.sort_values('avg_rank')
feature_scores.head(25)

## 8) Trực quan top đặc trưng

In [ ]:
plt.figure(figsize=(12, 8))
feature_scores.head(20)['tree_importance'].sort_values().plot(kind='barh')
plt.title('Top 20 đặc trưng theo Tree Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 9) Loại bớt biến trùng thông tin quá mạnh

Trong chuỗi thời gian, các biến như `water_level_lag1`, `water_level_lag2`, `water_level_lag3` thường rất giống nhau.
Vì vậy ta lấy top features trước, rồi loại bớt những biến có tương quan quá cao với nhau.

In [ ]:
def correlation_pruning(df, ranked_features, threshold=0.95, max_features=15):
    selected = []
    for feat in ranked_features:
        keep = True
        for chosen in selected:
            corr = df[feat].corr(df[chosen])
            if pd.notna(corr) and abs(corr) >= threshold:
                keep = False
                break
        if keep:
            selected.append(feat)
        if len(selected) >= max_features:
            break
    return selected

ranked_features = feature_scores.index.tolist()
selected_features_auto = correlation_pruning(
    model_df,
    ranked_features=ranked_features,
    threshold=0.95,
    max_features=15
)

selected_features_auto

## 10) Bộ đặc trưng khuyến nghị theo góc nhìn nghiệp vụ thủy văn

Ngoài bộ chọn tự động, nên giữ thêm một bộ **domain-driven** để dễ giải thích cho luận văn và mô hình.

Bộ sau cân bằng giữa:
- lịch sử mực nước,
- dòng vào / dòng ra,
- tình trạng dung tích,
- ngưỡng vận hành hồ,
- mùa vụ.

In [ ]:
selected_features_domain = [
    'water_level',
    'water_level_lag1',
    'water_level_lag3',
    'water_level_lag7',
    'water_level_roll_mean_3',
    'water_level_roll_mean_7',
    'water_level_roll_std_7',
    'inflow',
    'outflow',
    'net_inflow',
    'inflow_lag1',
    'inflow_lag3',
    'outflow_lag1',
    'storage',
    'storage_pct',
    'storage_pct_lag1',
    'normal_level',
    'dead_level',
    'level_above_normal',
    'month_sin',
    'month_cos'
]

selected_features_domain = [c for c in selected_features_domain if c in model_df.columns]
selected_features_domain

## 11) Xuất kết quả để dùng cho bước huấn luyện

Ta lưu ra 3 file:
1. `lake_feature_scores.csv`: bảng điểm toàn bộ đặc trưng.
2. `lake_selected_features_auto.txt`: danh sách chọn tự động.
3. `lake_model_data_selected.csv`: dataset chỉ giữ các cột cần cho mô hình.

In [ ]:
OUT_DIR = Path('/mnt/data/feature_selection_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

feature_scores.to_csv(OUT_DIR / 'lake_feature_scores.csv', encoding='utf-8-sig')

with open(OUT_DIR / 'lake_selected_features_auto.txt', 'w', encoding='utf-8') as f:
    for col in selected_features_auto:
        f.write(col + '
')

with open(OUT_DIR / 'lake_selected_features_domain.txt', 'w', encoding='utf-8') as f:
    for col in selected_features_domain:
        f.write(col + '
')

selected_dataset = model_df[['lake_code', 'lake_name', 'timestamp', target_col] + selected_features_domain].copy()
selected_dataset.to_csv(OUT_DIR / 'lake_model_data_selected.csv', index=False, encoding='utf-8-sig')

print('Đã lưu vào:', OUT_DIR)
print('Số đặc trưng domain-driven:', len(selected_features_domain))
print('Shape selected dataset:', selected_dataset.shape)

## 12) Kết luận nên dùng bộ nào?

Khuyến nghị thực tế:
- **Để chạy mô hình đầu tiên**: dùng `selected_features_domain` vì dễ giải thích.
- **Để tối ưu performance**: so sánh thêm với `selected_features_auto`.
- **Để viết luận văn**: trình bày cả hai hướng, sau đó chọn bộ có kết quả validation tốt hơn.

### Diễn giải nhanh
- Nhóm mạnh nhất thường là: `water_level`, các `lag` của `water_level`, các rolling mean/std.
- Nhóm kế tiếp: `inflow`, `outflow`, `net_inflow`, `storage_pct`, `storage`.
- Nhóm ngữ cảnh hồ: `normal_level`, `dead_level`.
- Nhóm mùa vụ: `month_sin`, `month_cos`, `doy_sin`, `doy_cos`.

### Lưu ý quan trọng
Nếu validation cho điểm quá cao, hãy kiểm tra:
1. có leakage hay không,
2. có đang dùng `water_level` hiện tại để dự báo `t+1` hay không,
3. mục tiêu nghiệp vụ là dự báo sau 1 ngày, 3 ngày hay 7 ngày.